In [ ]:
import numpy as np

import os, sys
current_dir = %pwd #os.path.dirname(__file__)
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)

import csv
import numpy as np
import itertools
import nltk

# from preprocessing import getSentenceData, getSentence
from rnnmodel import Model

In [ ]:
word_dim = 8000
hidden_dim = 100

unknown_token = "UNKNOWN_TOKEN"
sentence_start_token = "SENTENCE_START"
sentence_end_token = "SENTENCE_END"

vocabulary_size=8000

In [ ]:
# Read the data and append SENTENCE_START and SENTENCE_END tokens
print("Reading CSV file...")
with open('../data/reddit-comments-2015-08.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f, skipinitialspace=True)
    # Split full comments into sentences
    sentences = itertools.chain(*[nltk.sent_tokenize(x[0].lower()) for x in reader])
    # Append SENTENCE_START and SENTENCE_END
    sentences = ["%s %s %s" % (sentence_start_token, x, sentence_end_token) for x in sentences]
print("Parsed %d sentences." % (len(sentences)))

In [ ]:
# Tokenize the sentences into words
tokenized_sentences = [nltk.word_tokenize(sent) for sent in sentences]
# Filter the sentences having few words (including SENTENCE_START and SENTENCE_END)
tokenized_sentences = list(filter(lambda x: len(x) > 3, tokenized_sentences))

In [ ]:
# Count the word frequencies
word_freq = nltk.FreqDist(itertools.chain(*tokenized_sentences))
print("Found %d unique words tokens." % len(word_freq.items()))

In [ ]:
# Get the most common words and build index_to_word and word_to_index vectors
vocab = word_freq.most_common(vocabulary_size-1)
index_to_word = [x[0] for x in vocab]
index_to_word.append(unknown_token)
word_to_index = dict([(w,i) for i,w in enumerate(index_to_word)])

print("Using vocabulary size %d." % vocabulary_size)
print("The least frequent word in our vocabulary is '%s' and appeared %d times." % (vocab[-1][0], vocab[-1][1]))

In [ ]:
# Replace all words not in our vocabulary with the unknown token
for i, sent in enumerate(tokenized_sentences):
    tokenized_sentences[i] = [w if w in word_to_index else unknown_token for w in sent]

print("\nExample sentence: '%s'" % sentences[1])
print("\nExample sentence after Pre-processing: '%s'\n" % tokenized_sentences[0])

In [ ]:
# Create the training data
X_train = [[word_to_index[w] for w in sent[:-1]] for sent in tokenized_sentences]
y_train = [[word_to_index[w] for w in sent[1:]] for sent in tokenized_sentences]

# print("X_train shape: " + str(X_train.shape))
# print("y_train shape: " + str(y_train.shape))

In [ ]:
# Print an training data example
x_example, y_example = X_train[17], y_train[17]
print("x:\n%s\n%s" % (" ".join([index_to_word[x] for x in x_example]), x_example))
print("\ny:\n%s\n%s" % (" ".join([index_to_word[x] for x in y_example]), y_example))

In [ ]:
np.random.seed(10)
rnn = Model(word_dim, hidden_dim)

In [ ]:
# losses = rnn.train(X_train[:100], y_train[:100], learning_rate=0.005, nepoch=10, evaluate_loss_after=1)
losses = rnn.train(X_train, y_train, learning_rate=0.005, nepoch=10, evaluate_loss_after=1)

In [ ]:
trained_rnn = Model(word_dim, hidden_dim, model_path="./rnn_model.npz", toload=True)

In [ ]:
test_sentences = trained_rnn.predict(X_train[0])

In [ ]:
tokenized_test_sentence = [w if w in word_to_index else unknown_token for w in x_predict]

In [ ]:
print(test_sentences)
print(tokenized_test_sentence)